# Step 1. Congruency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/1_congruency.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**The question.** Does an irrelevant word interfere with naming a colour?

**Prediction.** Naming the ink is slower when the word names another colour (incongruent) than when it names the same one (congruent). The difference is the congruency effect.

**The analysis.** Response time on congruent trials against incongruent trials.

**The decision.** With three colours there are 3 congruent and 6 incongruent pairs of ink and word. Two wishes pull apart: *every pair equally often* (then one third of the trials is congruent) or *half of the trials congruent* (then the word tells something about the ink). This page builds both and counts them.

### The factors

A regular factor is set freely on every trial. The ink colour and the word are regular factors.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

Congruency is not set freely: it follows from the colour and the word. That is a derived factor.

In [ ]:
# a derived factor: it follows from colour and word on the same trial
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


congruency = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word])),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word])),
])

### Option A: every pair equally often

Colour and word are crossed, so each of the 9 pairs occurs equally often. `MinimumTrials(18)` asks for two rounds of the crossing.

In [ ]:
block_a = CrossBlock(design=[colour, word, congruency], crossing=[colour, word],
                     constraints=[MinimumTrials(18)])
trials_a = pd.DataFrame(synthesize_trials(block_a, 1, CMSGen)[0])
display(pd.crosstab(trials_a["colour"], trials_a["word"]))
display(trials_a["congruency"].value_counts().to_frame("trials"))

word,blue,green,red
colour,,,
blue,2,2,2
green,2,2,2
red,2,2,2


,trials
congruency,
incongruent,12
congruent,6


### Option B: half of the trials congruent

Cross the ink colour with congruency. Each ink colour is then congruent on half of its trials: 6 combinations, four rounds.

In [ ]:
block_b = CrossBlock(design=[colour, word, congruency], crossing=[colour, congruency],
                     constraints=[MinimumTrials(24)])
trials_b = pd.DataFrame(synthesize_trials(block_b, 1, CMSGen)[0])
display(pd.crosstab(trials_b["colour"], trials_b["congruency"]))
display(pd.crosstab(trials_b["colour"], trials_b["word"]))

congruency,congruent,incongruent
colour,,
blue,4,4
green,4,4
red,4,4


word,blue,green,red
colour,,,
blue,4,1,3
green,3,4,1
red,4,0,4


The first table is what was crossed: it is even. The second shows what was not: which word goes with an incongruent ink is left open, so those pairs can come out uneven. Run the cell again and compare.

### What does each option cost?

How often is a word shown in its own colour? One third means the word tells nothing about the ink.

In [ ]:
cost = pd.DataFrame({
    "congruent trials (%)": [round((t["congruency"] == "congruent").mean() * 100) for t in (trials_a, trials_b)],
    "word is in its own colour (%)": [round((t["word"] == t["colour"]).mean() * 100) for t in (trials_a, trials_b)],
}, index=["A: every pair equally often", "B: half congruent"])
display(cost)

,congruent trials (%),word is in its own colour (%)
A: every pair equally often,33,33
B: half congruent,50,50


**Check.** In which option could a participant guess the ink from the word? In which are there fewer congruent trials? Neither is free: which cost is acceptable depends on what you compare.

**Try.** Add a fourth colour to `colours` and run again. What happens to the share of congruent trials in option A?

**Next.** Step 2 keeps option B and asks what happened on the trial before.